# MindBridge 2.0: Crisis Risk Classification Model Training Pipeline
## Kaggle GPU Environment (`roberta-base`, Threshold Calibration & Quantized ONNX Export)

> **CLINICAL SAFEGUARD AND ETHICAL DISCLOSURE**
> * **Dataset-Derived Target**: The binary target (`crisis_label`: 0 for Non-Crisis, 1 for Crisis-Risk) is derived strictly from research annotations (`ihsansaad24/Mental-Health_Text-Classification_Dataset`) and **does not constitute a formal clinical diagnosis or validated suicide-risk assessment**.
> * **Clinical Nuance**: In psychiatric practice, individuals suffering from depression or anxiety may experience severe distress or self-harm impulses; labelling them 'Non-Crisis' is strictly a technical NLP convention to identify explicit crisis/suicidality text. Conversely, text labelled `Suicidal` frequently reflects passive ideation, philosophical thoughts, or historical recovery narratives without indicating imminent danger.
> * **Decoupled Architecture**: In production MindBridge 2.0, this machine-learning model functions as an early conversational risk screener. Emergency helplines, keyword/regex safety overrides, statutory compliance, and multi-dimensional PHQ-9 Item 9 protocols operate **deterministically** and are never bypassed or overridden by classifier probabilities.

### Kaggle Setup Instructions
1. **GPU Accelerator**: Ensure a GPU is enabled in the notebook settings:
   * Sidebar -> **Settings** -> **Accelerator** -> Select **GPU T4 x 2** or **GPU P100**.
2. **Input Dataset**: Upload the curated dataset splits (`train.csv`, `val.csv`, `test.csv`) as a Kaggle Dataset:
   * Sidebar -> **Input** -> **Add Input** -> Upload your dataset (e.g. named `mindbridge-curated-dataset`).
3. **Internet Access**: Ensure Internet is toggled **ON** in the sidebar settings so pretrained Hugging Face weights and pip packages can be downloaded.
4. **Execution**: Click **Save Version** -> **Save & Run All (Commit)** for a headless, reproducible training run.
5. **Output Retrieval**: After execution completes, download `mindbridge_crisis_model_deploy.zip` directly from the **Output** tab.

--- 
## 1. Kaggle Hardware & Accelerator Detection
Detect and report available Kaggle compute resources (GPU model, VRAM, CPU cores, RAM). Do not assume a GPU is present; verify and report actual hardware at runtime.

In [ ]:
import os
import sys
import psutil
import torch

print("=" * 60)
print("KAGGLE RUNTIME HARDWARE & ENVIRONMENT AUDIT")
print("=" * 60)
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CPU Cores Available: {os.cpu_count()}")
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"System RAM: {ram_gb:.2f} GB")

# Detect GPU hardware
cuda_available = torch.cuda.is_available()
if cuda_available:
    device_count = torch.cuda.device_count()
    primary_device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n[STATUS: GPU DETECTED]")
    print(f"  Primary Accelerator: {primary_device_name}")
    print(f"  Total GPU Devices:   {device_count}")
    print(f"  Primary VRAM:        {total_vram_gb:.2f} GB")
    print(f"  CUDA Capability:     {torch.cuda.get_device_capability(0)}")
    device = torch.device('cuda:0')
else:
    print(f"\n[STATUS: NO GPU DETECTED — CPU ONLY]")
    print("  ACTION REQUIRED FOR ACCELERATED TRAINING:")
    print("  Go to the right-hand panel -> 'Settings' -> 'Accelerator' -> Select 'GPU T4 x 2' or 'GPU P100'.")
    device = torch.device('cpu')

print(f"Selected Execution Device: {device}")
print("=" * 60)

--- 
## 2. Dependency Verification & Installation
Ensure all necessary pinned packages (Hugging Face Transformers, ONNX, ONNX Runtime, Scikit-learn) are installed and verified in the Kaggle environment.

In [ ]:
# Install / update necessary libraries inside Kaggle environment
!pip install -q \
    transformers==4.38.2 \
    accelerate==0.27.2 \
    datasets==2.17.1 \
    scikit-learn==1.4.1.post1 \
    onnx==1.15.0 \
    onnxruntime==1.17.1 \
    safetensors==0.4.2 \
    pyyaml==6.0.1

import random
import time
import json
import glob
import shutil
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, auc, roc_auc_score, confusion_matrix, accuracy_score
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

print("Installed Package Verification:")
import transformers
import sklearn
print(f"  Transformers: {transformers.__version__}")
print(f"  Scikit-Learn: {sklearn.__version__}")
print(f"  ONNX:         {onnx.__version__}")
print(f"  ONNX Runtime: {ort.__version__}")

--- 
## 3. Deterministic Seed Setup (Seed 42)
Enforce full reproducibility across Python, NumPy, PyTorch, and CUDA.

In [ ]:
SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)
print(f"Deterministic seed locked at: {SEED}")

--- 
## 4. Kaggle Directory Ingestion (`/kaggle/input/`)
Support Kaggle's `/kaggle/input/` directory structure. Automatically discover `train.csv`, `val.csv`, and `test.csv` across attached Kaggle datasets or local project folders.

In [ ]:
def find_dataset_files():
    """
    Discovers train.csv, val.csv, and test.csv under /kaggle/input/ or fallback directories.
    """
    search_roots = [
        '/kaggle/input',
        'data/processed',
        '.'
    ]
    
    train_candidates = []
    val_candidates = []
    test_candidates = []
    
    for root in search_roots:
        if os.path.exists(root):
            for dirpath, _, filenames in os.walk(root):
                for f in filenames:
                    if f == 'train.csv':
                        train_candidates.append(os.path.join(dirpath, f))
                    elif f == 'val.csv':
                        val_candidates.append(os.path.join(dirpath, f))
                    elif f == 'test.csv':
                        test_candidates.append(os.path.join(dirpath, f))
                        
    if not train_candidates or not val_candidates or not test_candidates:
        raise FileNotFoundError(
            f"Could not locate train.csv, val.csv, or test.csv in /kaggle/input/ or local paths.\n"
            f"Found trains: {train_candidates}\n"
            f"Found vals:   {val_candidates}\n"
            f"Found tests:  {test_candidates}\n"
            "Please ensure the dataset is attached to the Kaggle notebook (Input -> Add Input)."
        )
        
    return train_candidates[0], val_candidates[0], test_candidates[0]

train_path, val_path, test_path = find_dataset_files()
print(f"Discovered Ingestion Paths:")
print(f"  Train: {train_path}")
print(f"  Val:   {val_path}")
print(f"  Test:  {test_path}")

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

print(f"\nSplit Verification:")
print(f"  Train Samples: {len(train_df):,} | Non-Crisis: {(train_df['crisis_label'] == 0).sum():,} | Crisis-Risk: {(train_df['crisis_label'] == 1).sum():,}")
print(f"  Val Samples:   {len(val_df):,} | Non-Crisis: {(val_df['crisis_label'] == 0).sum():,} | Crisis-Risk: {(val_df['crisis_label'] == 1).sum():,}")
print(f"  Test Samples:  {len(test_df):,} | Non-Crisis: {(test_df['crisis_label'] == 0).sum():,} | Crisis-Risk: {(test_df['crisis_label'] == 1).sum():,}")

# Calculate positive class weight for imbalanced loss:
# pos_weight = negative_count / positive_count
num_neg = (train_df['crisis_label'] == 0).sum()
num_pos = (train_df['crisis_label'] == 1).sum()
pos_weight_value = num_neg / num_pos
print(f"Calculated Positive Class Loss Weight (pos_weight): {pos_weight_value:.4f}")

--- 
## 5. Tokenizer Configuration (`roberta-base`) & Dataset Building
Tokenize texts using `roberta-base` Byte-Pair Encoding with sequence length $L = 256$ (covering $94.88\%$ of samples without truncation).

In [ ]:
MODEL_NAME = 'roberta-base'
MAX_SEQ_LENGTH = 256
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Pretrained Tokenizer Loaded: {MODEL_NAME} | Vocab size: {tokenizer.vocab_size:,}")

class MentalHealthCrisisDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.float)
        }

train_dataset = MentalHealthCrisisDataset(train_df['text'].tolist(), train_df['crisis_label'].tolist(), tokenizer, MAX_SEQ_LENGTH)
val_dataset   = MentalHealthCrisisDataset(val_df['text'].tolist(), val_df['crisis_label'].tolist(), tokenizer, MAX_SEQ_LENGTH)
test_dataset  = MentalHealthCrisisDataset(test_df['text'].tolist(), test_df['crisis_label'].tolist(), tokenizer, MAX_SEQ_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True if cuda_available else False)
val_loader   = DataLoader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True if cuda_available else False)
test_loader  = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True if cuda_available else False)

print(f"DataLoaders Initialized: {len(train_loader)} train batches, {len(val_loader)} val batches, {len(test_loader)} test batches.")

--- 
## 6. Model Architecture & Weighted Loss Configuration
Instantiate `AutoModelForSequenceClassification` with 1 logit output. The positive class loss weight (`pos_weight = 3.329`) counteracts class imbalance by heavily penalizing false negatives.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    classifier_dropout=0.1
)
model.to(device)

# Weighted loss to counter class imbalance
pos_weight_tensor = torch.tensor([pos_weight_value], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# Hyperparameters
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TOTAL_STEPS = len(train_loader) * NUM_EPOCHS
WARMUP_STEPS = int(TOTAL_STEPS * 0.10)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=TOTAL_STEPS)

print(f"Total Model Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training Plan: {NUM_EPOCHS} epochs, {TOTAL_STEPS} optimization steps, {WARMUP_STEPS} warmup steps.")

--- 
## 7. Training Loop & Validation Checkpointing under `/kaggle/working/`
All checkpoints, tokenizer files, and metrics are written directly to Kaggle's writable directory: `/kaggle/working/`.

In [ ]:
# Target Kaggle's writable working directory
WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else './working'
CHECKPOINT_DIR = os.path.join(WORKING_DIR, 'best_checkpoint')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def evaluate_validation(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze(-1)
            loss = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            
    avg_loss = total_loss / len(loader.dataset)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    preds_50 = (all_probs >= 0.50).astype(int)
    prec_50 = precision_score(all_labels, preds_50, zero_division=0)
    rec_50  = recall_score(all_labels, preds_50, zero_division=0)
    f1_50   = f1_score(all_labels, preds_50, zero_division=0)
    
    precision_curve, recall_curve, _ = precision_recall_curve(all_labels, all_probs)
    pr_auc_val = auc(recall_curve, precision_curve)
    roc_auc_val = roc_auc_score(all_labels, all_probs)
    
    return {
        'loss': avg_loss,
        'pr_auc': pr_auc_val,
        'roc_auc': roc_auc_val,
        'prec_50': prec_50,
        'rec_50': rec_50,
        'f1_50': f1_50,
        'all_probs': all_probs,
        'all_labels': all_labels
    }

best_val_pr_auc = -1.0
best_val_results = None
training_log = []

print(f"Beginning model training loop (Output -> '{CHECKPOINT_DIR}')...")
for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    model.train()
    train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze(-1)
        loss = criterion(logits, labels)
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item() * len(labels)
        if (step + 1) % 150 == 0 or (step + 1) == len(train_loader):
            print(f"  Epoch {epoch}/{NUM_EPOCHS} | Batch {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    avg_train_loss = train_loss / len(train_loader.dataset)
    val_metrics = evaluate_validation(model, val_loader, criterion, device)
    elapsed = time.time() - start_time
    
    epoch_summary = {
        'epoch': epoch,
        'train_loss': round(float(avg_train_loss), 4),
        'val_loss': round(float(val_metrics['loss']), 4),
        'val_pr_auc': round(float(val_metrics['pr_auc']), 4),
        'val_roc_auc': round(float(val_metrics['roc_auc']), 4),
        'val_prec_50': round(float(val_metrics['prec_50']), 4),
        'val_rec_50': round(float(val_metrics['rec_50']), 4),
        'val_f1_50': round(float(val_metrics['f1_50']), 4),
        'elapsed_seconds': round(elapsed, 1)
    }
    training_log.append(epoch_summary)
    
    print(f"\n[EPOCH {epoch} SUMMARY ({elapsed:.1f}s)]")
    print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f}")
    print(f"  Val PR-AUC: {val_metrics['pr_auc']:.4f} | Val ROC-AUC: {val_metrics['roc_auc']:.4f}")
    print(f"  Val Metrics (@0.50): Prec={val_metrics['prec_50']:.4f}, Rec={val_metrics['rec_50']:.4f}, F1={val_metrics['f1_50']:.4f}")
    
    if val_metrics['pr_auc'] > best_val_pr_auc:
        best_val_pr_auc = val_metrics['pr_auc']
        best_val_results = val_metrics
        model.save_pretrained(CHECKPOINT_DIR)
        tokenizer.save_pretrained(CHECKPOINT_DIR)
        print(f"  >>> New best checkpoint saved to '{CHECKPOINT_DIR}' (Val PR-AUC: {best_val_pr_auc:.4f})")

# Save training log to working directory
with open(os.path.join(WORKING_DIR, 'training_history.json'), 'w') as f:
    json.dump(training_log, f, indent=2)

--- 
## 8. Validation Decision Threshold Calibration
Calibrate the decision threshold $\tau \in [0.10, 0.90]$ strictly on the validation set to meet safety requirements: Crisis Recall $\ge 0.95$ with maximum precision.

In [ ]:
# Load best checkpoint for validation calibration
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
model.to(device)
val_metrics = evaluate_validation(model, val_loader, criterion, device)

val_probs = val_metrics['all_probs']
val_labels = val_metrics['all_labels']

thresholds = np.arange(0.10, 0.92, 0.02)
threshold_results = []

for t in thresholds:
    preds = (val_probs >= t).astype(int)
    p = precision_score(val_labels, preds, zero_division=0)
    r = recall_score(val_labels, preds, zero_division=0)
    f = f1_score(val_labels, preds, zero_division=0)
    acc = accuracy_score(val_labels, preds)
    threshold_results.append({
        'threshold': round(float(t), 2),
        'precision': round(float(p), 4),
        'recall': round(float(r), 4),
        'f1': round(float(f), 4),
        'accuracy': round(float(acc), 4)
    })

df_sweep = pd.DataFrame(threshold_results)
print("Validation Decision Threshold Calibration Sweep:")
print(df_sweep.to_string(index=False))

# Target: Recall >= 0.95 with maximum precision
candidates = df_sweep[df_sweep['recall'] >= 0.95]
if len(candidates) > 0:
    best_row = candidates.sort_values(by='precision', ascending=False).iloc[0]
    CALIBRATED_THRESHOLD = float(best_row['threshold'])
    print(f"\nOptimal Calibrated Threshold Selected: {CALIBRATED_THRESHOLD} (Recall: {best_row['recall']:.4f}, Precision: {best_row['precision']:.4f}, F1: {best_row['f1']:.4f})")
else:
    best_row = df_sweep.sort_values(by='f1', ascending=False).iloc[0]
    CALIBRATED_THRESHOLD = float(best_row['threshold'])
    print(f"\nFallback Threshold Selected (Max F1): {CALIBRATED_THRESHOLD} (Recall: {best_row['recall']:.4f}, Precision: {best_row['precision']:.4f})")

with open(os.path.join(WORKING_DIR, 'calibrated_threshold.json'), 'w') as f:
    json.dump({
        'calibrated_threshold': CALIBRATED_THRESHOLD,
        'target_crisis_recall': 0.95,
        'val_metrics_at_calibrated_threshold': best_row.to_dict()
    }, f, indent=2)

--- 
## 9. Unbiased Test Set Evaluation (Strictly Post-Calibration)
Evaluate the frozen model and calibrated threshold $\tau^*$ on `test.csv` (unbiased generalization measurement).

In [ ]:
test_metrics = evaluate_validation(model, test_loader, criterion, device)
test_probs = test_metrics['all_probs']
test_labels = test_metrics['all_labels']

# Baseline 0.50
preds_50 = (test_probs >= 0.50).astype(int)
prec_50 = precision_score(test_labels, preds_50)
rec_50  = recall_score(test_labels, preds_50)
f1_50   = f1_score(test_labels, preds_50)
acc_50  = accuracy_score(test_labels, preds_50)

# Calibrated threshold
preds_cal = (test_probs >= CALIBRATED_THRESHOLD).astype(int)
prec_cal = precision_score(test_labels, preds_cal)
rec_cal  = recall_score(test_labels, preds_cal)
f1_cal   = f1_score(test_labels, preds_cal)
acc_cal  = accuracy_score(test_labels, preds_cal)
cm_cal   = confusion_matrix(test_labels, preds_cal)

print("=" * 60)
print(f"TEST EVALUATION REPORT (Total Samples: {len(test_labels):,})")
print("=" * 60)
print(f"PR-AUC:  {test_metrics['pr_auc']:.4f}")
print(f"ROC-AUC: {test_metrics['roc_auc']:.4f}")
print("-" * 60)
print(f"Baseline Threshold (0.50):")
print(f"  Accuracy:  {acc_50:.4f}")
print(f"  Precision: {prec_50:.4f}")
print(f"  Recall:    {rec_50:.4f}")
print(f"  F1-Score:  {f1_50:.4f}")
print("-" * 60)
print(f"Calibrated Safety Threshold ({CALIBRATED_THRESHOLD:.2f}):")
print(f"  Accuracy:  {acc_cal:.4f}")
print(f"  Precision: {prec_cal:.4f}")
print(f"  Recall:    {rec_cal:.4f}")
print(f"  F1-Score:  {f1_cal:.4f}")
print(f"Confusion Matrix (TN, FP / FN, TP):")
print(cm_cal)
print("=" * 60)

test_report = {
    'test_samples': len(test_labels),
    'pr_auc': float(test_metrics['pr_auc']),
    'roc_auc': float(test_metrics['roc_auc']),
    'baseline_threshold_0_50': {
        'accuracy': float(acc_50),
        'precision': float(prec_50),
        'recall': float(rec_50),
        'f1': float(f1_50)
    },
    'calibrated_threshold': {
        'threshold': float(CALIBRATED_THRESHOLD),
        'accuracy': float(acc_cal),
        'precision': float(prec_cal),
        'recall': float(rec_cal),
        'f1': float(f1_cal),
        'confusion_matrix': cm_cal.tolist()
    }
}
with open(os.path.join(WORKING_DIR, 'test_evaluation_report.json'), 'w') as f:
    json.dump(test_report, f, indent=2)

--- 
## 10. ONNX Export & Dynamic INT8 Quantization under `/kaggle/working/`
Export the fine-tuned model to ONNX format with dynamic batch and sequence axes, then quantize linear projections to INT8 for MindBridge CPU backend deployment.

In [ ]:
ONNX_FP32_PATH = os.path.join(WORKING_DIR, 'roberta_crisis_detector.onnx')
ONNX_INT8_PATH = os.path.join(WORKING_DIR, 'roberta_crisis_detector_quantized.onnx')

# Export using CPU model copy to guarantee clean graph trace
model.to('cpu')
model.eval()

dummy_input_ids = torch.randint(0, 1000, (1, MAX_SEQ_LENGTH), dtype=torch.long)
dummy_attention_mask = torch.ones((1, MAX_SEQ_LENGTH), dtype=torch.long)

print("Exporting PyTorch model to ONNX FP32...")
torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attention_mask),
    ONNX_FP32_PATH,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
        'logits': {0: 'batch_size'}
    },
    opset_version=17,
    do_constant_folding=True
)

onnx_model = onnx.load(ONNX_FP32_PATH)
onnx.checker.check_model(onnx_model)
fp32_size_mb = os.path.getsize(ONNX_FP32_PATH) / (1024 * 1024)
print(f"ONNX FP32 Model Verified: {fp32_size_mb:.2f} MB")

# Dynamic INT8 Quantization
print("Quantizing ONNX linear projections to INT8...")
quantize_dynamic(
    model_input=ONNX_FP32_PATH,
    model_output=ONNX_INT8_PATH,
    weight_type=QuantType.QInt8,
    op_types_to_quantize=['MatMul', 'Gemm']
)
int8_size_mb = os.path.getsize(ONNX_INT8_PATH) / (1024 * 1024)
print(f"ONNX INT8 Quantized Model: {int8_size_mb:.2f} MB (Compression: {fp32_size_mb / int8_size_mb:.2f}x)")

--- 
## 11. Numerical Consistency Check & CPU Benchmark
Verify consistency between PyTorch and ONNX INT8 ($|\hat{p}_{\text{PyTorch}} - \hat{p}_{\text{ONNX}}| \le 10^{-3}$ and label agreement $\ge 99.9\%$), and benchmark CPU latency.

In [ ]:
ort_options = ort.SessionOptions()
ort_options.intra_op_num_threads = 4
ort_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

ort_session = ort.InferenceSession(ONNX_INT8_PATH, ort_options, providers=['CPUExecutionProvider'])

# Consistency on 200 validation samples
test_subset_texts = val_df['text'].iloc[:200].tolist()
pytorch_probs = []
onnx_probs = []

for text in test_subset_texts:
    encoded = tokenizer(text, truncation=True, max_length=MAX_SEQ_LENGTH, padding='max_length', return_tensors='pt')
    
    with torch.no_grad():
        pt_out = model(encoded['input_ids'], encoded['attention_mask'])
        pt_prob = torch.sigmoid(pt_out.logits.squeeze(-1)).item()
        pytorch_probs.append(pt_prob)
        
    ort_inputs = {
        'input_ids': encoded['input_ids'].numpy(),
        'attention_mask': encoded['attention_mask'].numpy()
    }
    ort_out = ort_session.run(None, ort_inputs)
    onnx_logit = ort_out[0][0][0]
    onnx_prob = 1.0 / (1.0 + np.exp(-onnx_logit))
    onnx_probs.append(onnx_prob)

pytorch_probs = np.array(pytorch_probs)
onnx_probs = np.array(onnx_probs)
diffs = np.abs(pytorch_probs - onnx_probs)
max_diff = np.max(diffs)
mean_diff = np.mean(diffs)

pt_labels = (pytorch_probs >= CALIBRATED_THRESHOLD).astype(int)
onnx_labels = (onnx_probs >= CALIBRATED_THRESHOLD).astype(int)
agreement_pct = (pt_labels == onnx_labels).mean() * 100.0

print(f"Maximum Probability Discrepancy: {max_diff:.6f}")
print(f"Mean Probability Discrepancy:    {mean_diff:.6f}")
print(f"Classification Label Agreement:  {agreement_pct:.2f}%")

# Single-request CPU latency benchmark (50 iterations)
sample_enc = tokenizer("I am feeling completely overwhelmed and need help right now.", truncation=True, max_length=MAX_SEQ_LENGTH, padding='max_length', return_tensors='np')
sample_inputs = {'input_ids': sample_enc['input_ids'], 'attention_mask': sample_enc['attention_mask']}

# Warmup
for _ in range(5):
    ort_session.run(None, sample_inputs)

latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    ort_session.run(None, sample_inputs)
    latencies.append((time.perf_counter() - t0) * 1000.0)

print(f"\nCPU Inference Latency (Quantized INT8 ONNX):")
print(f"  p50 (Median): {np.percentile(latencies, 50):.2f} ms")
print(f"  p95:          {np.percentile(latencies, 95):.2f} ms")
print(f"  Mean:         {np.mean(latencies):.2f} ms +/- {np.std(latencies):.2f} ms")

--- 
## 12. Deployment Bundle Packaging & Kaggle Export Instructions
Package all deployment artifacts into a single archive (`mindbridge_crisis_model_deploy.zip`) in `/kaggle/working/` for direct download or versioning.

In [ ]:
DEPLOY_DIR = os.path.join(WORKING_DIR, 'mindbridge_crisis_model_deploy')
os.makedirs(DEPLOY_DIR, exist_ok=True)

# Copy artifacts to deployment directory
shutil.copy(ONNX_INT8_PATH, os.path.join(DEPLOY_DIR, 'crisis_classifier_quantized.onnx'))
shutil.copy(os.path.join(WORKING_DIR, 'calibrated_threshold.json'), os.path.join(DEPLOY_DIR, 'calibrated_threshold.json'))
shutil.copy(os.path.join(WORKING_DIR, 'test_evaluation_report.json'), os.path.join(DEPLOY_DIR, 'test_evaluation_report.json'))
shutil.copy(os.path.join(WORKING_DIR, 'training_history.json'), os.path.join(DEPLOY_DIR, 'training_history.json'))

# Save tokenizer files
tokenizer.save_pretrained(os.path.join(DEPLOY_DIR, 'tokenizer'))

# Create deployable zip archive in /kaggle/working/
zip_path = os.path.join(WORKING_DIR, 'mindbridge_crisis_model_deploy')
shutil.make_archive(zip_path, 'zip', DEPLOY_DIR)
final_zip = f"{zip_path}.zip"

zip_size_mb = os.path.getsize(final_zip) / (1024 * 1024)
print("=" * 60)
print(f"DEPLOYMENT ARCHIVE GENERATED SUCCESSFULLY:")
print(f"  File: {final_zip} ({zip_size_mb:.2f} MB)")
print("=" * 60)
print("\nHOW TO EXPORT / DOWNLOAD FROM KAGGLE:")
print("  Method 1 (Kaggle Web UI):")
print("    - After clicking 'Save Version' -> 'Save & Run All (Commit)', open the notebook viewer.")
print("    - Expand the right sidebar -> 'Output' section.")
print(f"    - Click the three dots next to '{os.path.basename(final_zip)}' -> 'Download'.")
print("\n  Method 2 (Create Kaggle Dataset from Output):")
print("    - In the Output section, click '+ New Dataset'.")
print("    - Name it 'mindbridge-crisis-model' to create a permanent versioned Kaggle dataset.")
print("\n  Method 3 (Kaggle CLI):")
print("    - Run: kaggle kernels output <username>/<notebook-slug> -p ./downloads")
print("=" * 60)